# Import Modules

In [190]:
#high level modules
import os
import imp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [191]:
# custom modules
universal_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/"

imp.load_source("universals", os.path.join(universal_dir, "universal_functions.py"))
from universals import load_pickle_file, calculate_vals

dump_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/DSS_Shiny/www/forecast/"


# Import models

In [192]:
model_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/operational_model/dump/five_ten/"

model_1 = load_pickle_file("model_1.pkl", model_dir)
model_2 = load_pickle_file("model_2.pkl", model_dir)
model_3 = load_pickle_file("model_3.pkl", model_dir)
model_4 = load_pickle_file("model_4.pkl", model_dir)

# Import data

In [193]:
data_dir = "/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/forecast_rollout/data/"

test = pd.read_csv(os.path.join(data_dir, "model_data.csv"))

test["date"] = pd.to_datetime(test["date"])



we also need the names of the features for the model to make sure the data is in the right format and order

In [194]:
feature_names = pd.read_csv(os.path.join(universal_dir, "data/training_1.csv")).columns

# remove date, and two target columns (first 3 columns)
feature_names = feature_names[3:]
feature_names

Index(['mean_1m_temp_degC_m1', 'mean_0_5m_temp_degC_m1',
       'total_solar_radiation', 'total_solar_radiation_m1',
       'total_solar_radiation_m2', 'mean_air_temp', 'min_air_temp',
       'max_air_temp', 'mean_air_temp_m1', 'min_air_temp_m1',
       'max_air_temp_m1', 'mean_rel_hum_m1', 'mean_rel_hum_m2', 'pump_cfs_m1',
       'pump_cfs_m2', 'pump_cfs_m3', 'mean_wind', 'max_wind', 'mean_wind_m1',
       'max_wind_m1', 'nf_cfs_m1', 'nf_cfs_m2', 'nf_cfs_m3', 'nf_cfs_m4',
       'chipmunk_cfs_m1', 'chipmunk_cfs_m2', 'chipmunk_cfs_m3',
       'chipmunk_cfs_m4'],
      dtype='object')

We're going to pick a few dates randomly to rollout the forecast, since that will be easier than doing all of them at once. 

In [195]:
one_date = pd.to_datetime("2024-7-15", format="%Y-%m-%d")
# create a date range of 7 days following
date_range = pd.date_range(start=one_date, periods=7, freq='D')

# filter the data (model_data)
filtered_data = test[(test['date'] >= date_range[0]) & (test['date'] <= date_range[-1])]

# make the date the index
filtered_data.set_index("date", inplace=True)

# and now coerce mean_1m_temp_degC and mean_0_5m_temp_degC to NaN no matter the value
filtered_data.loc[:, "mean_1m_temp_degC"] = np.nan
filtered_data.loc[:, "mean_0_5m_temp_degC"] = np.nan

# let's change the _m1 temp columns for date + 1 day and beyond to NaN
filtered_data.loc[filtered_data.index > one_date, "mean_1m_temp_degC_m1"] = np.nan
filtered_data.loc[filtered_data.index > one_date, "mean_0_5m_temp_degC_m1"] = np.nan

# create a new dataframe from filtered data
filtered_pulsing_pump = filtered_data.copy()
filtered_pulsing_pump = filtered_data.copy()

# and coerce any column starting with "pump" to NaN
for col in filtered_pulsing_pump.columns:
    if col.startswith("pump"):
        filtered_pulsing_pump.loc[:, col] = np.nan

# and coerce any column starting with "pump" to NaN
for col in filtered_pulsing_pump.columns:
    if col.startswith("pump"):
        filtered_pulsing_pump.loc[:, col] = np.nan


/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/3328975504.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data.loc[filtered_data.index > one_date, "mean_1m_temp_degC_m1"] = np.nan
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/3328975504.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_data.loc[filtered_data.index > one_date, "mean_0_5m_temp_degC_m1"] = np.nan


Great Now that we have a basic data set, we'll fill in the pump data with values for the next 7 days. These need to be reflective of the previous data, too, so we'll use the control data to fill in some of the data.

These are the columns for pumping:
'pump_cfs_m1', 'pump_cfs_m2', 'pump_cfs_m3'

First, we need transformed values to enter into the array. 

In [196]:
transformations = pd.read_csv("/Users/steeleb/Documents/GitHub/TLS_DSS/model_submodule/model_dev/data/mu_sigma.csv", index_col=0)
# get the index callled "pump_cfs"
x = transformations.loc[transformations.index == "pump_cfs", :].values
# calculate z score using the mean (first value) and the std (second value)
transformed_220 = (220 - x[0][0]) / x[0][1]
transformed_440 = (440 - x[0][0]) / x[0][1]

print(transformed_220, transformed_440)

0.22381065604391792 1.3177978994971216


In [197]:
# create a dataframe for static pump of 220, using the control data to manipulate the m1, m2, m3 columns.

# m1 for the first day will be m1 from the control data
filtered_static_pump.loc[filtered_static_pump.index == one_date, "pump_cfs_m1"] = filtered_data.loc[one_date, "pump_cfs_m1"]
# all of the m1 will be 220 cfs after the first day
mask = [isinstance(idx, pd.Timestamp) and idx > one_date for idx in filtered_static_pump.index]
filtered_static_pump.loc[mask, "pump_cfs_m1"] = transformed_220

# m2 for the first day will be m2 from the control data
filtered_static_pump.loc[filtered_static_pump.index == one_date, "pump_cfs_m2"] = filtered_data.loc[one_date, "pump_cfs_m2"]
# m2 for the second day will be m2 from the control data
filtered_static_pump.loc[filtered_static_pump.index == one_date + pd.Timedelta(days=1), "pump_cfs_m2"] = filtered_data.loc[one_date + pd.Timedelta(days=1), "pump_cfs_m2"]
# m2 will be 220 cfs after the second day
mask = [isinstance(idx, pd.Timestamp) and idx > one_date + pd.Timedelta(days=1) for idx in filtered_static_pump.index]
filtered_static_pump.loc[mask, "pump_cfs_m2"] = transformed_220

# m3 for the first day will be m3 from the control data
filtered_static_pump.loc[filtered_static_pump.index == one_date, "pump_cfs_m3"] = filtered_data.loc[one_date, "pump_cfs_m3"]
# m3 for the second day will be m3 from the control data
filtered_static_pump.loc[filtered_static_pump.index == one_date + pd.Timedelta(days=1), "pump_cfs_m3"] = filtered_data.loc[one_date + pd.Timedelta(days=1), "pump_cfs_m3"]
# m3 for the third day will be m3 from the control data
filtered_static_pump.loc[filtered_static_pump.index == one_date + pd.Timedelta(days=2), "pump_cfs_m3"] = filtered_data.loc[one_date + pd.Timedelta(days=2), "pump_cfs_m3"]
# m3 will be 220 cfs after the third day
mask = [isinstance(idx, pd.Timestamp) and idx > one_date + pd.Timedelta(days=2) for idx in filtered_static_pump.index]
filtered_static_pump.loc[mask, "pump_cfs_m3"] = transformed_220

# look at data to make sure it worked
print("Static Pump Data")
# each of the columns that start with pump
for col in filtered_static_pump.columns:
    if col.startswith("pump"):
        print(f"{col}: {filtered_static_pump[col].unique()}")


Static Pump Data
pump_cfs_m1: [1.56643136 0.22381066]
pump_cfs_m2: [1.29790722 1.56643136 0.22381066]
pump_cfs_m3: [1.68080276 1.29790722 1.56643136 0.22381066]


great. let's do the same as above, but the pump data will be based on the day of the week.

In [198]:
# create a column for day of week using the index in the pump_pulsing dataframe, in shorthand mon, tue, wed, etc.
filtered_pulsing_pump["day_of_week"] = filtered_pulsing_pump.index.day_name()

# m1 for the first day will be m1 from the control data
filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date, "pump_cfs_m1"] = filtered_data.loc[one_date, "pump_cfs_m1"]
# all of the m1 will be 220 cfs after the first day if a weekend, 440 cfs if a weekday
if filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date, "day_of_week"].values[0] in ["Saturday", "Sunday"]:
    # weekend
    filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date, "pump_cfs_m1"] = transformed_220
else:
    # weekday
    filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date, "pump_cfs_m1"] = transformed_440


# m2 for the first day will be m2 from the control data
filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date, "pump_cfs_m2"] = filtered_data.loc[one_date, "pump_cfs_m2"]
# m2 for the second day will be m2 from the control data
filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date + pd.Timedelta(days=1), "pump_cfs_m2"] = filtered_data.loc[one_date + pd.Timedelta(days=1), "pump_cfs_m2"]
# m2 will be 220 cfs after the second day if a weekend, 440 cfs if a weekday
if filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date + pd.Timedelta(days=1), "day_of_week"].values[0] in ["Saturday", "Sunday"]:
    # weekend
    filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date + pd.Timedelta(days=1), "pump_cfs_m2"] = transformed_220
else:
    # weekday
    filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date + pd.Timedelta(days=1), "pump_cfs_m2"] = transformed_440

# m3 for the first day will be m3 from the control data
filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date, "pump_cfs_m3"] = filtered_data.loc[one_date, "pump_cfs_m3"]
# m3 for the second day will be m3 from the control data
filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date + pd.Timedelta(days=1), "pump_cfs_m3"] = filtered_data.loc[one_date + pd.Timedelta(days=1), "pump_cfs_m3"]
# m3 for the third day will be m3 from the control data
filtered_pulsing_pump.loc[filtered_pulsing_pump.index == one_date + pd.Timedelta(days=2), "pump_cfs_m3"] = filtered_data.loc[one_date + pd.Timedelta(days=2), "pump_cfs_m3"]
# m3 will be 220 cfs after the third day if a weekend, 440 cfs if a weekday
if filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date + pd.Timedelta(days=2), "day_of_week"].values[0] in ["Saturday", "Sunday"]:
    # weekend
    filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date + pd.Timedelta(days=2), "pump_cfs_m3"] = transformed_220
else:
    # weekday
    filtered_pulsing_pump.loc[filtered_pulsing_pump.index > one_date + pd.Timedelta(days=2), "pump_cfs_m3"] = transformed_440

# look at data to make sure it worked
print("Pulsing Pump Data")
# each of the columns that start with pump
for col in filtered_pulsing_pump.columns:
    if col.startswith("pump"):
        print(f"{col}: {filtered_pulsing_pump[col].unique()}")

Pulsing Pump Data
pump_cfs_m1: [1.56643136 1.3177979 ]
pump_cfs_m2: [1.29790722 1.56643136 1.3177979 ]
pump_cfs_m3: [1.68080276 1.29790722 1.56643136 1.3177979 ]


Great. so now all columns should have values except the temp data. Let's look at the dataframes to make sure.

In [199]:
# check for NaNs using describe function
print(
filtered_data.describe(),
filtered_static_pump.describe(),
filtered_pulsing_pump.describe()
)


       mean_1m_temp_degC  mean_0_5m_temp_degC  mean_1m_temp_degC_m1  \
count                0.0                  0.0              1.000000   
mean                 NaN                  NaN              0.852248   
std                  NaN                  NaN                   NaN   
min                  NaN                  NaN              0.852248   
25%                  NaN                  NaN              0.852248   
50%                  NaN                  NaN              0.852248   
75%                  NaN                  NaN              0.852248   
max                  NaN                  NaN              0.852248   

       mean_0_5m_temp_degC_m1  total_solar_radiation  \
count                1.000000               7.000000   
mean                 0.263561              97.978195   
std                       NaN              25.075599   
min                  0.263561              72.313081   
25%                  0.263561              78.481135   
50%                  0.2

# Roll out forecast

Here, we're just iteratively modeling, for each of the 4 models for the next 7 days.

In [200]:
def make_forecast(features, model, model_number, valid_date):
    predictions = pd.DataFrame(columns=['mean_1m_temp_degC', 'mean_0_5m_temp_degC', 'model', 'valid_date'])
    # get the model name from the models object
    preds = {}
    preds = model.predict(features)
    temp_df = pd.DataFrame(preds, columns=['mean_1m_temp_degC', 'mean_0_5m_temp_degC'])
    for col in temp_df.columns:
        temp_df[col] = (temp_df[col])
    temp_df['model'] = model_number
    temp_df['valid_date'] = valid_date
    predictions = pd.concat([predictions, temp_df], ignore_index=True)
    return predictions

# and remove data that is not needed for modeling
control_for_model = filtered_data.drop(columns = ["mean_1m_temp_degC", "mean_0_5m_temp_degC"])

static_for_model = filtered_static_pump.drop(columns = ["mean_1m_temp_degC", "mean_0_5m_temp_degC"])
pulsing_for_model = filtered_pulsing_pump.drop(columns = ["day_of_week", "mean_1m_temp_degC", "mean_0_5m_temp_degC"])

# we also need to make sure the columns are in the same order as the model was trained on
control_for_model = control_for_model[feature_names]
static_for_model = static_for_model[feature_names]
pulsing_for_model = pulsing_for_model[feature_names]


In [201]:
# grab first day of data
data_for_model = control_for_model.loc[control_for_model.index == one_date, :]

forecasted_temp_collated = pd.DataFrame(columns=['mean_1m_temp_degC', 'mean_0_5m_temp_degC', 'model', 'valid_date'])

for model in [model_1, model_2, model_3, model_4]:
    if model == model_1:
        model_number = "1"
    elif model == model_2:
        model_number = "2"
    elif model == model_3:
        model_number = "3"
    elif model == model_4:
        model_number = "4"
    forecasted_temp = (make_forecast(data_for_model, 
                                model,
                                model_number,
                                one_date))
    forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)    

# run each model for the 7 day forecast
for model in [1, 2, 3, 4]:
    
    for day in [1, 2, 3, 4, 5, 6]:
        fore_date = one_date + pd.Timedelta(days=day)
        # grab the data for the model
        data_for_model = control_for_model.loc[control_for_model.index == fore_date, :]
        # and now we can make the forecast, but we want the model to be the same as the one we are using
        if model == 1:
            # we need the value where the model is 1 and the valid date is the same as the forecast date from the forecasted_temp_collated file
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "1") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "1") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_1, "1", fore_date)
        elif model == 2:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "2") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "2") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_2, "2", fore_date)
        elif model == 3:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "3") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "3") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_3, "3", fore_date)
        elif model == 4:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "4") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "4") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_4, "4", fore_date)
        
        # now store the forecasted temp in the dataframe by adding a row from the forecasted temp
        forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)


/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/26517464.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions = pd.concat([predictions, temp_df], ignore_index=True)
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/1137597954.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/26517464.py

In [202]:
# we need to back transform the data for mean 1m and mean 0.5m
forecasted_temp_collated["mean_1m_temp_degC"] = forecasted_temp_collated["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
forecasted_temp_collated["mean_0_5m_temp_degC"] = forecasted_temp_collated["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

# save the files to csv
forecasted_temp_collated.to_csv(os.path.join(dump_dir, "forecasted_temp_control_collated.csv"), index=False)

In [203]:
# grab first day of data
data_for_model = static_for_model.loc[static_for_model.index == one_date, :]

forecasted_temp_collated = pd.DataFrame(columns=['mean_1m_temp_degC', 'mean_0_5m_temp_degC', 'model', 'valid_date'])

for model in [model_1, model_2, model_3, model_4]:
    if model == model_1:
        model_number = "1"
    elif model == model_2:
        model_number = "2"
    elif model == model_3:
        model_number = "3"
    elif model == model_4:
        model_number = "4"
    forecasted_temp = (make_forecast(data_for_model, 
                                model,
                                model_number,
                                one_date))
    forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)    

# run each model for the 7 day forecast
for model in [1, 2, 3, 4]:
    
    for day in [1, 2, 3, 4, 5, 6]:
        fore_date = one_date + pd.Timedelta(days=day)
        # grab the data for the model
        data_for_model = static_for_model.loc[static_for_model.index == fore_date, :]
        # and now we can make the forecast, but we want the model to be the same as the one we are using
        if model == 1:
            # we need the value where the model is 1 and the valid date is the same as the forecast date from the forecasted_temp_collated file
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "1") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "1") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_1, "1", fore_date)
        elif model == 2:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "2") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "2") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_2, "2", fore_date)
        elif model == 3:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "3") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "3") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_3, "3", fore_date)
        elif model == 4:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "4") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "4") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_4, "4", fore_date)
        
        # now store the forecasted temp in the dataframe by adding a row from the forecasted temp
        forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)


/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/26517464.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions = pd.concat([predictions, temp_df], ignore_index=True)
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/870525271.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/26517464.py:

In [204]:
# we need to back transform the data for mean 1m and mean 0.5m
forecasted_temp_collated["mean_1m_temp_degC"] = forecasted_temp_collated["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
forecasted_temp_collated["mean_0_5m_temp_degC"] = forecasted_temp_collated["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

# save the files to csv
forecasted_temp_collated.to_csv(os.path.join(dump_dir, "forecasted_temp_static_collated.csv"), index=False)

In [205]:
# grab first day of data
data_for_model = pulsing_for_model.loc[pulsing_for_model.index == one_date, :]

forecasted_temp_collated = pd.DataFrame(columns=['mean_1m_temp_degC', 'mean_0_5m_temp_degC', 'model', 'valid_date'])

for model in [model_1, model_2, model_3, model_4]:
    if model == model_1:
        model_number = "1"
    elif model == model_2:
        model_number = "2"
    elif model == model_3:
        model_number = "3"
    elif model == model_4:
        model_number = "4"
    forecasted_temp = (make_forecast(data_for_model, 
                                model,
                                model_number,
                                one_date))
    forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)    

# run each model for the 7 day forecast
for model in [1, 2, 3, 4]:
    
    for day in [1, 2, 3, 4, 5, 6]:
        fore_date = one_date + pd.Timedelta(days=day)
        # grab the data for the model
        data_for_model = pulsing_for_model.loc[pulsing_for_model.index == fore_date, :]
        # and now we can make the forecast, but we want the model to be the same as the one we are using
        if model == 1:
            # we need the value where the model is 1 and the valid date is the same as the forecast date from the forecasted_temp_collated file
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "1") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "1") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_1, "1", fore_date)
        elif model == 2:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "2") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "2") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_2, "2", fore_date)
        elif model == 3:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "3") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "3") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_3, "3", fore_date)
        elif model == 4:
            data_for_model["mean_1m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "4") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_1m_temp_degC"].values
            data_for_model["mean_0_5m_temp_degC_m1"] = forecasted_temp_collated.loc[(forecasted_temp_collated["model"] == "4") & (forecasted_temp_collated["valid_date"] == fore_date - pd.Timedelta(days=1)), "mean_0_5m_temp_degC"].values
            forecasted_temp = make_forecast(data_for_model, model_4, "4", fore_date)
        
        # now store the forecasted temp in the dataframe by adding a row from the forecasted temp
        forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)



/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/26517464.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions = pd.concat([predictions, temp_df], ignore_index=True)
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/2629456554.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  forecasted_temp_collated = pd.concat([forecasted_temp_collated, forecasted_temp], ignore_index=True)
/var/folders/x8/v7bmckc139v2dqcm04jgfjlr0000gn/T/ipykernel_91256/26517464.py

In [206]:
# we need to back transform the data for mean 1m and mean 0.5m
forecasted_temp_collated["mean_1m_temp_degC"] = forecasted_temp_collated["mean_1m_temp_degC"] * transformations.loc["mean_1m_temp_degC", "sd"] + transformations.loc["mean_1m_temp_degC", "mean"]
forecasted_temp_collated["mean_0_5m_temp_degC"] = forecasted_temp_collated["mean_0_5m_temp_degC"] * transformations.loc["mean_0_5m_temp_degC", "sd"] + transformations.loc["mean_0_5m_temp_degC", "mean"]

# save the files to csv
forecasted_temp_collated.to_csv(os.path.join(dump_dir, "forecasted_temp_pulsing_collated.csv"), index=False)